In [1]:
# Load Packages 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from functools import reduce
from plotnine import * 
import statsmodels.formula.api as smf 
from pathlib import Path 

In [2]:
# Setting up directory 

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "code"/ "data"
OUTPUT_DIR = PROJECT_DIR / "output"

OUTPUT_DIR.mkdir(exist_ok=True) 

In [3]:
# Merge Diagnostics Helper Function 
def merge_with_diagnostics(left, right, on, how, name, validate=None, suffixes=("_x", "_y")):
    before_left = len(left)
    before_right = len(right)
    merged = left.merge(
        right,
        on=on,
        how=how,
        indicator=True,
        validate=validate,
        suffixes=suffixes)
    print(f"\n{name}")
    print(f"Left rows before merge: {before_left}")
    print(f"Right rows before merge: {before_right}")
    print(f"Rows after merge: {len(merged)}")
    print(merged["_merge"].value_counts())
    return merged.drop(columns="_merge")

In [4]:
# Loading previously created datasets 
cas_2018_2022 = pd.read_csv(DATA_DIR/"cas_refugees_by_commune_2018_2022.csv") 
merged_elections = pd.read_csv(DATA_DIR/"merged_elections.csv")

In [5]:
# Loading ISTAT's Employment Dataset 

unemp_2018 = pd.read_csv(DATA_DIR/"unemp_data_2018_with_regione.csv") 
unemp_2019 = pd.read_csv(DATA_DIR/"unemp_data_2019_with_regione.csv") 
unemp_2021 = pd.read_csv(DATA_DIR/"unemp_data_2021_with_regione.csv") 
unemp_2022 = pd.read_csv(DATA_DIR/"unemp_data_2022_with_regione.csv") 

In [6]:
# Merging the datasets across the years 

unemp_2018_2019 = merge_with_diagnostics(
    unemp_2018,
    unemp_2019,
    on=["comune", "Regione"],
    how="left",
    name="Merge 2018 and 2019 unemployment data",
    suffixes=("_2018", "_2019"))

unemp_2018_2019


Merge 2018 and 2019 unemployment data
Left rows before merge: 7903
Right rows before merge: 7903
Rows after merge: 7903
_merge
both          7903
left_only        0
right_only       0
Name: count, dtype: int64


,comune,employment_rate_2018,Regione,employment_rate_2019
0,Agliè,72.83,PIEMONTE,72.70
1,Airasca,71.29,PIEMONTE,71.22
2,Ala di Stura,73.41,PIEMONTE,75.00
3,Albiano d'Ivrea,69.54,PIEMONTE,71.71
4,Almese,72.66,PIEMONTE,72.68
...,...,...,...,...
7898,Villaputzu,54.35,SARDEGNA,54.70
7899,Villasalto,58.32,SARDEGNA,58.53
7900,Villasimius,61.38,SARDEGNA,62.83
7901,Villasor,53.05,SARDEGNA,53.57


In [7]:
keys = ["comune", "Regione"]

unemp_2021_renamed = unemp_2021.rename(
    columns={col: f"{col}_2021" for col in unemp_2021.columns if col not in keys})

unemp_2018_2021 = merge_with_diagnostics(
    unemp_2018_2019,
    unemp_2021_renamed,
    on=keys,
    how="left",
    name="Merge 2018-2019 and 2021 unemployment data")

unemp_2018_2021


Merge 2018-2019 and 2021 unemployment data
Left rows before merge: 7903
Right rows before merge: 7903
Rows after merge: 7903
_merge
both          7903
left_only        0
right_only       0
Name: count, dtype: int64


,comune,employment_rate_2018,Regione,employment_rate_2019,employment_rate_2021
0,Agliè,72.83,PIEMONTE,72.70,73.28
1,Airasca,71.29,PIEMONTE,71.22,72.09
2,Ala di Stura,73.41,PIEMONTE,75.00,68.93
3,Albiano d'Ivrea,69.54,PIEMONTE,71.71,73.25
4,Almese,72.66,PIEMONTE,72.68,74.24
...,...,...,...,...,...
7898,Villaputzu,54.35,SARDEGNA,54.70,55.96
7899,Villasalto,58.32,SARDEGNA,58.53,59.33
7900,Villasimius,61.38,SARDEGNA,62.83,61.46
7901,Villasor,53.05,SARDEGNA,53.57,57.00


In [8]:
unemp_2022_renamed = unemp_2022.rename(
    columns={col: f"{col}_2022" for col in unemp_2022.columns if col not in keys})

unemp_2018_2022 = merge_with_diagnostics(
    unemp_2018_2021,
    unemp_2022_renamed,
    on=keys,
    how="left",
    name="Merge 2018-2021 and 2022 unemployment data")

unemp_2018_2022


Merge 2018-2021 and 2022 unemployment data
Left rows before merge: 7903
Right rows before merge: 7903
Rows after merge: 7903
_merge
both          7903
left_only        0
right_only       0
Name: count, dtype: int64


,comune,employment_rate_2018,Regione,employment_rate_2019,employment_rate_2021,employment_rate_2022
0,Agliè,72.83,PIEMONTE,72.70,73.28,72.26
1,Airasca,71.29,PIEMONTE,71.22,72.09,70.97
2,Ala di Stura,73.41,PIEMONTE,75.00,68.93,70.11
3,Albiano d'Ivrea,69.54,PIEMONTE,71.71,73.25,71.81
4,Almese,72.66,PIEMONTE,72.68,74.24,73.44
...,...,...,...,...,...,...
7898,Villaputzu,54.35,SARDEGNA,54.70,55.96,56.78
7899,Villasalto,58.32,SARDEGNA,58.53,59.33,60.57
7900,Villasimius,61.38,SARDEGNA,62.83,61.46,63.75
7901,Villasor,53.05,SARDEGNA,53.57,57.00,58.26


In [9]:
# Calculating the mean employment rate for every commune between 2018 to 2022 

rate_emp = ["employment_rate_2018",
    "employment_rate_2019",
    "employment_rate_2021",
    "employment_rate_2022"] 

unemp_2018_2022["employment_rate_mean"] = (
    unemp_2018_2022[rate_emp].mean(axis=1))

In [10]:
# Renaming the column so that it is easier to join the datasets 
unemp_2018_2022.rename(columns={"comune": "COMUNE"}, inplace = True)

In [11]:
# Creating a data frame for regression by merging election data and refugee data

reg_data = merge_with_diagnostics(
    merged_elections,
    cas_2018_2022[["comune_id", "COMUNE", "refugees_per_1000_inhabitants_mean"]],
    on = "COMUNE",
    how = "left",
    name = "Merge elections with refugee data")


Merge elections with refugee data
Left rows before merge: 7951
Right rows before merge: 2939
Rows after merge: 7951
_merge
left_only     5114
both          2837
right_only       0
Name: count, dtype: int64


In [12]:
# Standardizing column names in the regression data frame so that it is easier to merge 

reg_data["COMUNE_upper"] = (
    reg_data["COMUNE"]
    .astype(str)
    .str.strip()
    .str.upper())

unemp_2018_2022["COMUNE_upper"] = (
    unemp_2018_2022["COMUNE"]
    .astype(str)
    .str.strip()
    .str.upper()) 

In [13]:
# Creating the already created data frame for regression with commune level employment data

reg_data_2 = merge_with_diagnostics(
    reg_data,
    unemp_2018_2022[["COMUNE_upper", "employment_rate_mean"]],
    on="COMUNE_upper",
    how="left",
    name="Merge regression data with mean employment rate")

reg_data_2


Merge regression data with mean employment rate
Left rows before merge: 7951
Right rows before merge: 7903
Rows after merge: 7956
_merge
both          7690
left_only      266
right_only       0
Name: count, dtype: int64


,COMUNE,CIRCOSCRIZIONE_x,ELETTORI,ELETTORI_MASCHI,VOTANTI,VOTANTI_MASCHI,SCHEDE_BIANCHE,+EUROPA_x,10 VOLTE MEGLIO,AUTODETERMINATZIONE,...,centre_right_coalition_perc_22,far_right_coalition_perc_22,centre_left_change_22_18,centre_right_change_22_18,far_right_change_22_18,far_left_change_22_18,comune_id,refugees_per_1000_inhabitants_mean,COMUNE_upper,employment_rate_mean
0,ABANO TERME,0.0,15576,7331,12359,5944,134,343.0,47.0,0.0,...,5.841101,40.630271,-5.293704,-5.551407,9.940085,2.760763,3499.0,0.353054,ABANO TERME,70.5075
1,ABBADIA CERRETO,0.0,232,110,198,97,2,1.0,0.0,0.0,...,10.526316,45.029240,-5.316321,-1.089846,9.170654,1.169591,NaN,NaN,ABBADIA CERRETO,74.3825
2,ABBADIA LARIANA,0.0,2598,1268,2062,1015,44,54.0,2.0,0.0,...,6.229686,39.382449,-5.850382,-3.809111,11.351411,4.767064,NaN,NaN,ABBADIA LARIANA,73.5200
3,ABBADIA SAN SALVATORE,0.0,4957,2340,3766,1876,50,76.0,0.0,0.0,...,3.588441,29.374405,-6.684898,-4.271357,11.849179,3.429660,4635.0,6.603622,ABBADIA SAN SALVATORE,72.1125
4,ABBASANTA,0.0,2240,1103,1521,757,10,18.0,0.0,30.0,...,7.774141,33.960720,-16.361017,-7.150251,17.524165,3.682488,NaN,NaN,ABBASANTA,60.3950
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7951,ZUGLIO,0.0,494,244,346,175,5,5.0,2.0,0.0,...,5.303030,50.000000,-18.383255,0.389736,14.450867,3.030303,NaN,NaN,ZUGLIO,69.2875
7952,ZUMAGLIA,0.0,869,412,676,322,13,26.0,0.0,0.0,...,6.859206,46.570397,-3.400231,-7.933694,12.250870,4.151625,1118.0,2.545825,ZUMAGLIA,72.6700
7953,ZUMPANO,0.0,1964,960,1485,773,13,7.0,0.0,0.0,...,11.482085,20.846906,4.376227,-0.706467,15.594380,1.058632,NaN,NaN,ZUMPANO,57.4800
7954,ZUNGOLI,0.0,933,454,668,346,18,1.0,3.0,0.0,...,6.639004,24.066390,-5.641414,-0.846026,17.030462,1.037344,NaN,NaN,ZUNGOLI,59.6925


In [14]:
# Creating and running the linear regression with centre-left vote share change as dependent variable and refugees per 1000 inhabitants 
# as independent variable while the mean employment rate and centre-left vote share in 2018 are controls 

model_controls = smf.ols(
    "centre_left_change_22_18 ~ refugees_per_1000_inhabitants_mean + employment_rate_mean + centre_left_coalition_perc_18",
    data=reg_data_2, missing="drop"
).fit()

print(model_controls.summary()) 

                               OLS Regression Results                               
Dep. Variable:     centre_left_change_22_18   R-squared:                       0.150
Model:                                  OLS   Adj. R-squared:                  0.149
Method:                       Least Squares   F-statistic:                     166.0
Date:                      Sun, 07 Jun 2026   Prob (F-statistic):           4.46e-99
Time:                              23:14:02   Log-Likelihood:                -8683.8
No. Observations:                      2822   AIC:                         1.738e+04
Df Residuals:                          2818   BIC:                         1.740e+04
Df Model:                                 3                                         
Covariance Type:                  nonrobust                                         
                                         coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------

In [17]:
# Scatterplot of Centre-Left Vote Share Change vs. Refugees per 1,000 Inhabitants

graph_plot = (ggplot(reg_data_2,
        aes(x="refugees_per_1000_inhabitants_mean",
            y="centre_left_change_22_18"))
    + geom_point(alpha=0.5, color="#264C48")
    + geom_smooth(method="lm", se=True, color="#264C48")
    + labs(title="Scatterplot of Centre-Left Vote Share Change vs. Refugees per 1,000 Inhabitants",
        x="Refugees per 1,000 Inhabitants",
        y="Centre-Left Vote Share Change, 2022–2018")
    + theme_minimal())

graph_plot 

graph_plot.save(
    OUTPUT_DIR / "figure_1.png",
    width=8,
    height=5,
    dpi=300)

/opt/anaconda3/lib/python3.13/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 5 in image.
/opt/anaconda3/lib/python3.13/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /Users/hrishikroy/Desktop/qss_20_refugee/output/figure_1.png
/opt/anaconda3/lib/python3.13/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_point : Removed 5127 rows containing missing values.
